# Land Registry & House Prices

This notebook explores the availability of data through HM Land Registry (covers England and Wales) as well as the Scottish and Northern Irish equivalents, to see what columns feature in each of the datasets, and whether or not these sources will be suitable to feature as part of this project.

In [ ]:
import pandas as pd
import requests
from pathlib import Path
import time

NOTEBOOK_DIR = Path.cwd()
LOCAL_DATA_DIR = NOTEBOOK_DIR / "datasets"
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_DIR = NOTEBOOK_DIR / "profiles"
PROFILE_DIR.mkdir(parents=True, exist_ok=True)
FULL_SUMMARY_DIR = NOTEBOOK_DIR / "full_summaries"
FULL_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

# HM Land Registry

The first of the data sources that I explore is the Land Registry, as this contains several datasets that appear worth exploring as part of this work.

The data available:
- Geospatial Data: https://www.gov.uk/government/publications/geospatial-commission-data-catalogue-hm-land-registry 
    - Download: https://assets.publishing.service.gov.uk/media/5f75db788fa8f54e9b91877f/geospatial-data-catalogue-hm-land-registry.csv 
- Index Polygons (Open Source): https://use-land-property-data.service.gov.uk/datasets/inspire
    - Several download links, find another way
- Land charges & spatial: https://use-land-property-data.service.gov.uk/datasets/llc
    - Several download links, find another way
- Overseas companies owning property in England and Wales: https://use-land-property-data.service.gov.uk/datasets/ocod
    - Feature in API, (/datasets is full endpoint - may iterate through same way as ONS?) https://use-land-property-data.service.gov.uk/api/v1/ 
- UK companies with property in England and Wales: https://use-land-property-data.service.gov.uk/datasets/ccod
    - Feature in API
- Price paid data: https://www.gov.uk/government/collections/price-paid-data
    - Feature in API
- Info requests: https://use-land-property-data.service.gov.uk/datasets/av_req
    - Feature in API
- Info requests: https://use-land-property-data.service.gov.uk/datasets/rfi
    - Feature in API
- Transaction data: https://use-land-property-data.service.gov.uk/datasets/td
    - Feature in API
- HPI reports (cover whole UK): https://www.gov.uk/government/collections/uk-house-price-index-reports
    - https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/UK-HPI-full-file-2026-05.csv (have to assume this format is consistent each month), which are also made up of other sub tables worth keeping, under the same
    HPI schema (or something like that):
    - https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Average-prices-2026-05.csv
    - https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Average-prices-Property-Type-2026-05.csv
    - https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Sales-2026-05.csv
    - https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Cash-mortgage-sales-2026-05.csv
    - https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/First-Time-Buyer-Former-Owner-Occupied-2026-05.csv
    - https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/New-and-Old-2026-05.csv
    - https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Indices-2026-05.csv
    - https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Indices-seasonally-adjusted-2026-05.csv
    - https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Average-price-seasonally-adjusted-2026-05.csv
    - https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Repossession-2026-05.csv



In [ ]:
LAND_REGISTRY_DATASETS = [
    {
        "dataset_id": "geospatial-data-catalogue-hm-land-registry",
        "url": "https://assets.publishing.service.gov.uk/media/5f75db788fa8f54e9b91877f/geospatial-data-catalogue-hm-land-registry.csv",
    },
    {
        "dataset_id": "hpi-UK-full-file-2026-05",
        "url": "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/UK-HPI-full-file-2026-05.csv",
    },
    {
        "dataset_id": "hpi-average-prices-2026-05",
        "url": "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Average-prices-2026-05.csv",
    },
    {
        "dataset_id": "hpi-average-prices-property-type-2026-05",
        "url": "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Average-prices-Property-Type-2026-05.csv",
    },
    {
        "dataset_id": "hpi-sales-2026-05",
        "url": "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Sales-2026-05.csv",
    },
    {
        "dataset_id": "hpi-cash-mortgage-sales-2026-05",
        "url": "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Cash-mortgage-sales-2026-05.csv",
    },
    {
        "dataset_id": "hpi-FTB-former-owner-occupied-2026-05",
        "url": "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/First-Time-Buyer-Former-Owner-Occupied-2026-05.csv",
    },
    {
        "dataset_id": "hpi-new-and-old-2026-05",
        "url": "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/New-and-Old-2026-05.csv",
    },
    {
        "dataset_id": "hpi-indices-2026-05",
        "url": "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Indices-2026-05.csv",
    },
    {
        "dataset_id": "hpi-indices-seasonally-adjusted-2026-05",
        "url": "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Indices-seasonally-adjusted-2026-05.csv",
    },
    {
        "dataset_id": "hpi-average-price-seasonally-adjusted-2026-05",
        "url": "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Average-price-seasonally-adjusted-2026-05.csv",
    },
    {
        "dataset_id": "hpi-repossession-2026-05",
        "url": "https://publicdata.landregistry.gov.uk/market-trend-data/house-price-index-data/Repossession-2026-05.csv",
    },
]


def save_downloaded_dataset(dataset_id: str, download_url: str) -> Path:
    dataset_dir = LOCAL_DATA_DIR / dataset_id
    dataset_dir.mkdir(parents=True, exist_ok=True)

    local_path = dataset_dir / "raw.csv"
    time.sleep(0.5)
    response = requests.get(download_url, timeout=120)
    response.raise_for_status()
    local_path.write_bytes(response.content)
    print(f"Saved raw file to: {local_path}")
    return local_path


def load_dataset_from_path(file_path: Path) -> pd.DataFrame:
    suffix = file_path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(file_path, low_memory=False)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(file_path)
    raise ValueError(f"Unsupported file type: {file_path}")


def build_full_column_summary(dataset_id: str, df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column in df.columns:
        s = df[column]
        rows.append(
            {
                "dataset_id": dataset_id,
                "column": column,
                "dtype": str(s.dtype),
                "row_count": int(len(df)),
                "null_count": int(s.isna().sum()),
                "null_pct": round(float(s.isna().mean() * 100), 2),
                "unique_count": int(s.nunique(dropna=True)),
                "min": s.min() if pd.api.types.is_numeric_dtype(s) else None,
                "max": s.max() if pd.api.types.is_numeric_dtype(s) else None,
                "mean": round(float(s.mean()), 4) if pd.api.types.is_numeric_dtype(s) else None,
                "median": round(float(s.median()), 4) if pd.api.types.is_numeric_dtype(s) else None,
            }
        )
    return pd.DataFrame(rows)


def build_full_categorical_breakdown(dataset_id: str, df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    categorical_cols = df.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    for col in categorical_cols:
        s = df[col].dropna()
        value_counts = s.value_counts(dropna=False)
        for value, count in value_counts.items():
            rows.append(
                {
                    "dataset_id": dataset_id,
                    "column": col,
                    "value": value,
                    "count": int(count),
                    "pct_of_rows": round(float(count / len(df) * 100), 4),
                }
            )
    return pd.DataFrame(rows)


def save_full_summaries(dataset_id: str, df: pd.DataFrame) -> tuple[Path, Path]:
    column_summary = build_full_column_summary(dataset_id, df)
    column_path = FULL_SUMMARY_DIR / f"{dataset_id}_column_summary.csv"
    column_summary.to_csv(column_path, index=False)

    categorical_breakdown = build_full_categorical_breakdown(dataset_id, df)
    categorical_path = FULL_SUMMARY_DIR / f"{dataset_id}_categorical_breakdown.csv"
    categorical_breakdown.to_csv(categorical_path, index=False)

    print(f"Saved full column summary to: {column_path}")
    print(f"Saved full categorical breakdown to: {categorical_path}")
    return column_path, categorical_path


def save_profile(dataset_id: str, df: pd.DataFrame) -> Path:
    profile = {
        "dataset_id": dataset_id,
        "row_count": int(len(df)),
        "column_count": int(len(df.columns)),
        "missing_cells": int(df.isna().sum().sum()),
        "missing_pct": round(float(df.isna().sum().sum() / (len(df) * max(len(df.columns), 1)) * 100), 4),
        "numeric_columns": int(len(df.select_dtypes(include=["number"]).columns)),
        "categorical_columns": int(len(df.select_dtypes(include=["object", "category", "string"]).columns)),
    }
    profile_path = PROFILE_DIR / f"{dataset_id}_profile.csv"
    pd.DataFrame([profile]).to_csv(profile_path, index=False)
    print(f"Saved profile to: {profile_path}")
    return profile_path


for dataset in LAND_REGISTRY_DATASETS:
    dataset_id = dataset["dataset_id"]
    url = dataset["url"]
    print(f"\nProcessing: {dataset_id}")

    try:
        raw_path = save_downloaded_dataset(dataset_id, url)
        df = load_dataset_from_path(raw_path)
        save_full_summaries(dataset_id, df)
        save_profile(dataset_id, df)
        print(f"Complete: {dataset_id} ({len(df)} rows, {len(df.columns)} columns)")
    except Exception as exc:
        print(f"ERROR processing {dataset_id}: {exc}")

# Registers of Scotland

The second section explores Scottish data, not found in the Land Registry.

Data (not all available):
- Sales (£1465pm): https://www.ros.gov.uk/data-and-statistics/sales-and-bespoke-data/sales-data-reports
- Country of origin (can be requested): https://www.ros.gov.uk/data-and-statistics/sales-and-bespoke-data/country-of-origin-company-data-set
- House price stats: https://www.ros.gov.uk/__data/assets/excel_doc/0007/258847/ros_all_stats_June_2026.xlsx 
- Small areas: https://www.ros.gov.uk/data-and-statistics/property-market-statistics/small-area-statistics

# Land Property Services (NI)

The final UK country for exploration in this notebook is Northern Ireland. This data is quite well contained all on the single LPS page, pointing to lots of sources: https://www.finance-ni.gov.uk/articles/land-property-services-lps-data-available-online

Data available for locating through this link:
- Pensioner allowance and Disabled persons allowance
- Property Vacancy rates
- NI House Price index
- New dwellings (new housing I presume?)
- Housing stock 2008 - 2023
- Valuation Lists
- OSNI spatial
- OSNI open data which contains 121 datasets 
- Annual End of Year Rating Balances by District Council and Sector
- LPS Call handling
- Info requests
- Complaints & Correspondance 
- Town centre database (appears to be dead URL)
- Rates paid by District Council and Sector